# 03 - Baselines classiques : HOG + SVM / Random Forest / k-NN

> **Projet AutoMaze AI** 
- classification de 6 usagers de la route, entraînée de zéro.
> Règles du cours : pas de YOLO, pas de détecteur, **pas de poids préentraînés** (`pretrained=True` interdit).
> Les cellules `TODO` sont à écrire par l'équipe. Ne supprimez pas les questions : répondez-y en dessous, elles nourrissent le rapport.

**Objectif :** obtenir rapidement un **score de référence** sur la validation. Le CNN du notebook 04 devra le battre.

**Pourquoi c'est important :**
- si la baseline est proche du hasard (1/6 ≈ 17 %), le problème vient probablement des données ;
- sans point de comparaison, un score de CNN ne veut rien dire pour le jury.

**HOG en une phrase :** on découpe l'image en petites cellules et on compte, dans chacune, les directions des contours.
Le SVM apprend ensuite une frontière entre les classes dans cet espace de descripteurs.
**Comparaison demandée par le descriptif du projet :** plusieurs classifieurs de ML classique (SVM, Random Forest, k-NN) après extraction de caractéristiques (HOG). Les trois utilisent les mêmes descripteurs et le même jeu de validation, pour une comparaison équitable.

## 0. Imports

In [ ]:
import time

import pandas as pd
import matplotlib.pyplot as plt
import cv2
from sklearn.svm import LinearSVC, SVC                 # classifieurs SVM (linéaire = rapide, RBF = plus souple)
from sklearn.preprocessing import StandardScaler       # mettre les descripteurs à la même échelle
from sklearn.pipeline import make_pipeline             # enchaîner normalisation + classifieur proprement
from sklearn.dummy import DummyClassifier              # score plancher (hasard / classe majoritaire)
from sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier    # forêt d'arbres de décision
from sklearn.neighbors import KNeighborsClassifier     # k plus proches voisins
from sklearn.decomposition import PCA                  # réduire la dimension avant k-NN (HOG est de grande dimension)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
# --- Configuration commune (identique dans chaque notebook) ---
from pathlib import Path
import random

import numpy as np

# Exécution sur Google Colab ou en local : le même notebook fonctionne dans les deux cas.
try:
    import google.colab  # type: ignore  # noqa: F401  (module présent uniquement sur Colab : avertissement Pylance normal en local)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Sur Colab, tout ce qui est écrit dans /content disparaît à la fin de la session.
    # On travaille donc dans Google Drive : données, figures et poids y sont conservés.
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/AutoMaze_Ai")  # TODO(équipe) : adapter au dossier partagé de l'équipe
else:
    # En local, ce notebook est dans notebooks/ : la racine du projet est le dossier parent.
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Colab" if IN_COLAB else "Local", "| racine du projet :", ROOT)
DATA_DIR = ROOT / "data"          # données locales, ignorées par Git
RAW_DIR = DATA_DIR / "raw"        # fichiers téléchargés tels quels (ex. annotations COCO)
CROPS_DIR = DATA_DIR / "crops"    # recadrages produits par le notebook 02
MODELS_DIR = ROOT / "models"      # poids + métadonnées (les .pt sont ignorés par Git)
FIGURES_DIR = ROOT / "reports" / "figures"  # figures à reprendre dans le rapport

for d in (RAW_DIR, CROPS_DIR, MODELS_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Ordre des classes = source de vérité pour TOUT le projet (notebooks + simulateur).
# Ne le changez plus une fois l'entraînement commencé : l'indice 0 doit toujours désigner la même classe.
CLASSES = ["person", "bicycle", "car", "motorcycle", "bus", "truck"]
CLASSES_FR = ["Personne", "Vélo", "Voiture", "Moto", "Bus", "Camion"]

# Graine aléatoire : rend les résultats reproductibles (même découpage, mêmes initialisations).
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 1. Charger les recadrages train et validation

On **n'utilise pas** `data/crops/test` ici.

In [ ]:
# TODO(équipe) : fonction load_split(split) -> (X_images, y) qui lit data/crops/<split>/<classe>/*.jpg
# y = indice de la classe dans CLASSES (respecter cet ordre !).

## 2. Extraire les descripteurs HOG

`cv2.HOGDescriptor` suffit (pas besoin d'installer scikit-image). Paramètres à comprendre et à faire varier :
taille de fenêtre, de bloc, de cellule et nombre d'orientations.

**Question :** HOG utilise-t-il la couleur ? Qu'est-ce que cela implique pour distinguer un bus d'un camion ?

In [ ]:
# TODO(équipe) : créer un HOGDescriptor adapté à INPUT_SIZE, calculer les descripteurs de train et de val.
# Afficher la dimension d'un descripteur.

## 3. Score plancher

In [ ]:
# TODO(équipe) : DummyClassifier(strategy="most_frequent") puis "stratified" : F1 macro sur la validation.

## 4. Entraîner le SVM

Commencez par `LinearSVC`, puis essayez `SVC(kernel="rbf")` si le temps le permet. Notez chaque essai dans le journal des runs du rapport.

In [ ]:
# TODO(équipe) : make_pipeline(StandardScaler(), LinearSVC(C=...)), entraînement, mesure du temps.
# Faire varier C (ex. 0.01, 0.1, 1, 10) et garder le meilleur selon le F1 macro de VALIDATION.
# Utiliser la fonction evaluer(...) définie dans la section 4 bis pour alimenter le tableau comparatif
# (exécuter d'abord la cellule qui la définit, ou la déplacer ici).

## 4 bis. Random Forest

Même descripteurs HOG que le SVM. Un Random Forest n'a pas besoin de normalisation, mais on garde le même pipeline d'entrée pour comparer à armes égales.

**Question :** pourquoi un Random Forest est-il souvent moins à l'aise qu'un SVM sur des descripteurs HOG ? (indice : dimension, corrélation entre descripteurs)

In [ ]:
# Hypothèses : X_train_hog, y_train, X_val_hog, y_val viennent des sections 1 et 2 (à écrire par l'équipe).
# Journal des runs : chaque essai ajoute un dictionnaire, repris dans le tableau comparatif de la section 4 quater.
runs = []

def evaluer(nom, modele, X_tr, y_tr, X_va, y_va):
    """Entraîne, mesure les temps et les métriques de validation, stocke le run."""
    t0 = time.time(); modele.fit(X_tr, y_tr); t_fit = time.time() - t0
    t0 = time.time(); y_pred = modele.predict(X_va); t_pred = time.time() - t0
    p, r, f1, _ = precision_recall_fscore_support(y_va, y_pred, average="macro", zero_division=0)
    run = {"modèle": nom, "accuracy": accuracy_score(y_va, y_pred), "precision (macro)": p,
           "recall (macro)": r, "F1 (macro)": f1, "temps entraînement (s)": t_fit, "temps prédiction (s)": t_pred}
    runs.append(run)
    print(f"{nom}: F1 macro = {f1:.3f} | accuracy = {run['accuracy']:.3f} | entraînement {t_fit:.1f} s")
    return modele, y_pred

# Faire varier n_estimators (ex. 100, 300) et max_depth (None, 20) ; class_weight="balanced" aide si les classes sont déséquilibrées.
rf, y_pred_rf = evaluer(
    "Random Forest",
    RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=SEED),
    X_train_hog, y_train, X_val_hog, y_val,
)

## 4 ter. k-NN

k-NN ne « apprend » rien : il compare chaque image de validation aux images d'entraînement. En grande dimension les distances deviennent peu discriminantes ; on réduit donc d'abord avec une PCA (ajustée sur le train uniquement).

**Question :** que se passe-t-il si on ajuste la PCA ou le StandardScaler sur train + validation ? (indice : fuite de données)

In [ ]:
# Faire varier k (ex. 3, 5, 11, 21) et le nombre de composantes PCA (ex. 50, 100, 200).
# Garder la meilleure combinaison selon le F1 macro de VALIDATION, jamais selon le test.
for k in (3, 5, 11, 21):
    knn = make_pipeline(StandardScaler(), PCA(n_components=100, random_state=SEED),
                        KNeighborsClassifier(n_neighbors=k, weights="distance", n_jobs=-1))
    evaluer(f"k-NN (k={k})", knn, X_train_hog, y_train, X_val_hog, y_val)

## 4 quater. Tableau comparatif

À reprendre dans la section « Comparaison des modèles » du rapport. Ajoutez ici les runs du SVM (section 4) avec la même fonction `evaluer`.

In [ ]:
comparaison = pd.DataFrame(runs).sort_values("F1 (macro)", ascending=False).reset_index(drop=True)
display(comparaison.round(3))
comparaison.to_csv(FIGURES_DIR / "comparaison_modeles_classiques.csv", index=False)

# Matrice de confusion (normalisée par ligne) du meilleur modèle : quelles classes sont les plus difficiles ?
meilleur = comparaison.loc[0, "modèle"]
print("Meilleur modèle classique :", meilleur)

## 5. Analyse

In [ ]:
# TODO(équipe) : classification_report et matrice de confusion (normalisée par ligne) sur la validation.
# Sauvegarder la figure dans FIGURES_DIR.

## 6. Conclusions

- Meilleur F1 macro de validation, par modèle (SVM / Random Forest / k-NN) : …
- Modèle classique retenu comme baseline pour le CNN : …
- Classes les mieux / les moins bien reconnues : …
- Confusions principales et hypothèse d'explication : …

Ces résultats vont dans la ligne « Baseline » du journal des runs (section 4 du rapport) et dans la comparaison des modèles.